In [8]:
import os
import sys
import yaml
import torch
import jiwer
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.recognizer.model import CRNN, ctc_greedy_decode
from src.dataset.loader import get_dataloaders
from src.trainer.train_cer import CharsetCodec
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'Arial']

In [9]:
# 1. Nạp Config và Dữ liệu Validation
with open('../configs/default.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

target_h = config['preprocess']['target_h']
charset_path = '../' + config['paths']['charset']
val_dir = '../' + config['paths']['synthetic_val']
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

codec = CharsetCodec(charset_path)

from src.dataset.loader import OCRDataset, collate_fn
from torch.utils.data import DataLoader

val_ds = OCRDataset([val_dir], charset_path, is_train=False, target_h=target_h)
val_loader = DataLoader(
    val_ds, batch_size=64, shuffle=False,
    collate_fn=collate_fn, num_workers=0,
    pin_memory=True
)

print('Số ảnh tập Val:', len(val_ds))
print('Kích thước tập ký tự:', len(codec.charset))


Dataset: 200 samples, train=False
Số ảnh tập Val: 200
Kích thước tập ký tự: 248


In [ ]:
device = torch.device('cpu')

In [ ]:
# 2. Khởi tạo 3 Model từ Checkpoint
def load_model(ckpt_path):
    model = CRNN(
        num_classes=len(codec),
        lstm_hidden=config['model']['lstm_hidden'],
        lstm_layers=config['model']['lstm_layers'],
        lstm_dropout=0.0 # Bỏ dropout lúc Test
    ).to(device)
    
    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
        model.eval()
        return model
    else:
        print(f'Không tìm thấy file: {ckpt_path}')
        return None

models = {
    'Phase 1 (F1)': load_model('../checkpoints/crnn_best_f1.pth'),
    'Phase 2 (F2)': load_model('../checkpoints/crnn_best_f2.pth'),
    'Phase 3 (F3)': load_model('../checkpoints/crnn_best_f3.pth')
}

# 3. Chạy Suy Luận trên tập Validation
results = []
for batch in tqdm(val_loader, desc='Đang dự đoán'):
    images = batch['image'].to(device)
    targets = batch['label']
    
    with torch.no_grad():
        preds = {}
        for name, m in models.items():
            if m is not None:
                out = m(images)
                preds[name] = ctc_greedy_decode(out, codec.charset)
                
    for i in range(len(targets)):
        row = {'Nhãn Thực Tế (Target)': targets[i]}
        for name in models.keys():
            if models[name] is not None:
                row[name] = preds[name][i]
        results.append(row)

df = pd.DataFrame(results)

# 4. Tính toán CER & Vẽ Biểu Đồ
cers = {}
for name in models.keys():
    if models[name] is not None:
        valid_targets = []
        valid_preds = []
        for t, p in zip(df['Nhãn Thực Tế (Target)'], df[name]):
            if len(t.strip()) > 0:
                valid_targets.append(t)
                valid_preds.append(p if len(p.strip()) > 0 else ' ')
        try:
            cer = jiwer.cer(valid_targets, valid_preds) * 100
        except Exception:
            cer = 100.0
        cers[name] = cer

if cers:
    plt.figure(figsize=(8, 5))
    bars = plt.bar(list(cers.keys()), list(cers.values()), color=['#FF9999', '#66B2FF', '#99FF99'])
    plt.title('So sánh Lỗi Ký Tự (CER %) qua 3 Checkpoints', fontsize=14, fontweight='bold')
    plt.ylabel('CER (%) - Càng thấp càng tốt')
    plt.ylim(0, max(cers.values()) + 5)
    
    for bar in bars:
        yval = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2, yval + 0.5, f'{yval:.2f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')
        
    plt.show()

# 5. Xem song song kết quả của 10 mẫu ngẫu nhiên
display(df.sample(min(15, len(df))))


Đang dự đoán:   0%|          | 0/4 [00:00<?, ?it/s]

: 